# Phase 3 RR with the V3 Phase 1 checkpoint

This is a separate Phase 3 notebook for the **third Phase 1 architecture, V3**. It loads `v3_model.pt` and does not run Phase 1 pretraining. The V3 transformer and tokenizer definitions below are copied from the supplied V3 notebook so the checkpoint uses its original parameter names and shapes.

The study is deliberately small: **LoRA, rsLoRA, LoRMA+, and one smaller-student distillation run**. The three PEFT runs share the same data, context window, classification head, and training schedule. The student is initialized from six evenly spaced V3 layers and is distilled from the best PEFT teacher chosen on development Macro-F1. No test labels are used for model selection.

Attach one Kaggle dataset containing the six Phase 3 JSON files, `label_vocab.json`, `bpe_en_25k.json`, and the supplied `v3_model.pt`. Run from top to bottom. All Phase 3 outputs go to `/kaggle/working/phase3_rr_v3/`.

**Reference point:** the previously reported V1 public score was 0.4311. This is a historical public score, not a V3 development score and not a promised result.


## 1. Runtime and inputs

In [ ]:
import os, json, math, time, random, gc, zipfile, copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/phase3_rr_v3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
if DEVICE.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

def find_asset(filename):
    hits = sorted(INPUT_ROOT.rglob(filename))
    if not hits:
        raise FileNotFoundError(f"{filename} was not found under {INPUT_ROOT}. Attach the Kaggle dataset containing it.")
    if len(hits) > 1:
        sizes = {p.stat().st_size for p in hits}
        if len(sizes) != 1:
            raise RuntimeError(f"Multiple different files named {filename}: {hits}")
        print(f"Found duplicate copies of {filename}; using {hits[0]}")
    return hits[0]

ASSET_NAMES = ["CL_train.json", "CL_dev.json", "CL_test.json", "IT_train.json", "IT_dev.json", "IT_test.json", "label_vocab.json", "bpe_en_25k.json", "v3_model.pt"]
ASSETS = {name: find_asset(name) for name in ASSET_NAMES}
for name, path in ASSETS.items(): print(f"{name}: {path} ({path.stat().st_size / (1024**2):.1f} MiB)")


## 2. Load the official label vocabulary and Phase 3 splits

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

raw_vocab = read_json(ASSETS["label_vocab.json"])
if isinstance(raw_vocab, dict) and all(isinstance(v, int) for v in raw_vocab.values()):
    LABELS = [name for name, _ in sorted(raw_vocab.items(), key=lambda item: item[1])]
elif isinstance(raw_vocab, list):
    LABELS = list(raw_vocab)
else:
    raise ValueError("Unsupported label_vocab.json format; inspect it before training.")
if len(LABELS) != len(set(LABELS)): raise ValueError("Duplicate official labels.")
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}
print("Labels:", LABELS)

def normalize_documents(obj, split_name):
    if isinstance(obj, list): return obj
    if isinstance(obj, dict):
        docs=[]
        for doc_id, value in obj.items():
            if isinstance(value, dict):
                docs.append({"id": str(doc_id), **value})
            elif isinstance(value, list) and split_name.endswith("_test.json"):
                # Unlabeled test files use {document_id: [sentence, ...]}.
                docs.append({"id": str(doc_id), "text": value})
            else:
                raise ValueError(f"{split_name}: unsupported record for document {doc_id}")
        return docs
    raise TypeError(f"{split_name} must be a list or ID-to-document mapping")

SPLIT_NAMES = ["CL_train.json", "CL_dev.json", "CL_test.json", "IT_train.json", "IT_dev.json", "IT_test.json"]
SPLITS = {name: normalize_documents(read_json(ASSETS[name]), name) for name in SPLIT_NAMES}
def validate_split(name, docs, labeled):
    ids=[]; n_sent=0
    for i, doc in enumerate(docs):
        if not isinstance(doc, dict) or "id" not in doc or not isinstance(doc.get("text"), list): raise ValueError(f"{name}[{i}] must have id and a text list")
        if not all(isinstance(s, str) for s in doc["text"]): raise TypeError(f"{name}[{i}].text must contain strings")
        ids.append(str(doc["id"])); n_sent += len(doc["text"])
        if labeled:
            labs=doc.get("labels")
            if not isinstance(labs, list) or len(labs) != len(doc["text"]): raise ValueError(f"{name}[{i}] does not have one label per sentence")
            unknown=set(labs)-set(LABELS)
            if unknown: raise ValueError(f"{name}[{i}] has unknown labels: {unknown}")
    if len(ids) != len(set(ids)): raise ValueError(f"Duplicate document IDs in {name}")
    print(f"{name}: {len(docs)} docs, {n_sent} sentences")
for name, docs in SPLITS.items(): validate_split(name, docs, not name.endswith("_test.json"))


## 3. Load the exact V3 BPE tokenizer

In [ ]:
BPE_PATH = ASSETS["bpe_en_25k.json"]
bpe_data = read_json(BPE_PATH)
bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = int(bpe_data["vocab_size"])
assert bpe_vocab_size == 25_000 and len(bpe_itos) == 25_000
bpe_stoi = {token: idx for idx, token in enumerate(bpe_itos)}
merge_to_id = {}
for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge
    merge_to_id[(bpe_stoi[left_token], bpe_stoi[right_token])] = 261 + merge_index
PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]


In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

## 4. Prepare sentence-centered context examples

In [ ]:
# A target sentence is encoded together with its immediate document neighbors.
# Neighbor sentences are truncated more aggressively so the target keeps most of its tokens.
MAX_TARGET_TOKENS = 192
MAX_NEIGHBOR_TOKENS = 64
BATCH_SIZE = 16
EPOCHS = 4
PATIENCE = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16.0
ADAPTER_DROPOUT = 0.05
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.02
KD_TEMPERATURE = 2.0
KD_SOFT_WEIGHT = 0.5

DOMAIN_TO_ID = {"CL": 0, "IT": 1}
def prepare_examples(domain, docs, labeled):
    rows=[]
    for doc in docs:
        doc_id=str(doc["id"]); sents=doc["text"]
        tokenized=[tokenizer.encode(s) or [UNK_TOKEN_ID] for s in sents]
        for i, ids in enumerate(tokenized):
            current=ids[:MAX_TARGET_TOKENS]
            left=tokenized[i-1][-MAX_NEIGHBOR_TOKENS:] if i>0 else []
            right=tokenized[i+1][:MAX_NEIGHBOR_TOKENS] if i+1<len(tokenized) else []
            prefix=[BOS_TOKEN_ID]+left+[EOS_TOKEN_ID]
            focus_start=len(prefix)
            tokens=prefix+current+[EOS_TOKEN_ID]+right+[EOS_TOKEN_ID]
            label_id=LABEL_TO_ID[doc["labels"][i]] if labeled else -100
            rows.append({"input_ids":tokens,"focus_start":focus_start,"focus_end":focus_start+len(current),"label":label_id,"domain_id":DOMAIN_TO_ID[domain],"domain":domain,"doc_id":doc_id,"sentence_index":i,"document_length":len(sents)})
    return rows

TRAIN_EXAMPLES=[]; DEV_EXAMPLES=[]; TEST_EXAMPLES=[]
for dom in ("CL","IT"):
    TRAIN_EXAMPLES += prepare_examples(dom, SPLITS[f"{dom}_train.json"], True)
    DEV_EXAMPLES += prepare_examples(dom, SPLITS[f"{dom}_dev.json"], True)
    TEST_EXAMPLES += prepare_examples(dom, SPLITS[f"{dom}_test.json"], False)
print("Sentence examples:",len(TRAIN_EXAMPLES),"train /",len(DEV_EXAMPLES),"dev /",len(TEST_EXAMPLES),"test")
train_counts=Counter(x["label"] for x in TRAIN_EXAMPLES)
print("Train supports:",{ID_TO_LABEL[k]:v for k,v in sorted(train_counts.items())})

class SentenceExampleDataset(Dataset):
    def __init__(self, rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        item=dict(self.rows[index]); item["example_index"]=index; return item

def collate_examples(batch):
    ids=[torch.tensor(x["input_ids"],dtype=torch.long) for x in batch]
    input_ids=pad_sequence(ids,batch_first=True,padding_value=PAD_TOKEN_ID)
    focus=torch.zeros_like(input_ids,dtype=torch.bool)
    for i,x in enumerate(batch): focus[i,x["focus_start"]:x["focus_end"]]=True
    return {"input_ids":input_ids,"focus_mask":focus,"labels":torch.tensor([x["label"] for x in batch],dtype=torch.long),"domain_ids":torch.tensor([x["domain_id"] for x in batch],dtype=torch.long),"domains":[x["domain"] for x in batch],"indices":torch.tensor([x["example_index"] for x in batch],dtype=torch.long)}

def make_loader(rows, shuffle):
    return DataLoader(SentenceExampleDataset(rows),batch_size=BATCH_SIZE,shuffle=shuffle,collate_fn=collate_examples,num_workers=2,pin_memory=(DEVICE.type=="cuda"))

TRAIN_LOADER=make_loader(TRAIN_EXAMPLES,True)
DEV_LOADER=make_loader(DEV_EXAMPLES,False)
TEST_LOADER=make_loader(TEST_EXAMPLES,False)


## 5. V3 Phase 1 transformer definition

The next cell uses the supplied V3 model class with the same parameter names as the Phase 1 checkpoint. Phase 1 pretraining code and its optimizer/training loop are intentionally not included, so running this notebook cannot restart Phase 1 training.

In [ ]:
# 9. V3 Encoder-Only Prefix-LM
#
# V3 uses a single encoder-style Transformer stack.
#
# Attention pattern:
#   Prefix tokens  -> bidirectional attention
#   Target tokens  -> prefix + previous target tokens
#
# This allows the model to perform autoregressive
# continuation without a separate decoder or cross-attention.
#
# V3 uses:
# - Encoder-only Prefix-LM
# - Standard Multi-Head Attention
# - Learned positional embeddings
# - RMSNorm
# - Prefix-bidirectional + target-causal attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation
# - Tied token embeddings and LM head
#
# This is intentionally different from both V1 and V2.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-6
    ):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Prefix-LM Attention
# ---------------------------------------------------------

class V3MultiHeadAttention(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):

        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):

        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        attention_mask=None,
        past_key_value=None,
        use_cache=False
    ):

        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(x)
        )

        v = self._split_heads(
            self.v_proj(x)
        )

        if past_key_value is not None:

            past_k, past_v = past_key_value

            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Prefix-LM Attention Mask
# ---------------------------------------------------------

def create_prefix_lm_mask(
    sequence_length,
    prefix_length,
    device
):
    """
    Creates the attention pattern for Prefix-LM.

    Prefix:
        Every prefix token can attend to every
        other prefix token.

    Target:
        Every target token can attend to:
        - all prefix tokens
        - itself
        - previous target tokens

    Future target tokens are masked.
    """

    mask = torch.zeros(
        sequence_length,
        sequence_length,
        device=device,
        dtype=torch.bool
    )

    # Prefix tokens attend bidirectionally
    mask[
        :prefix_length,
        :prefix_length
    ] = True

    # Target tokens attend to all prefix tokens
    mask[
        prefix_length:,
        :prefix_length
    ] = True

    # Target tokens attend causally to target tokens
    target_length = (
        sequence_length - prefix_length
    )

    causal_target_mask = torch.tril(
        torch.ones(
            target_length,
            target_length,
            device=device,
            dtype=torch.bool
        )
    )

    mask[
        prefix_length:,
        prefix_length:
    ] = causal_target_mask

    return mask


# ---------------------------------------------------------
# GELU Feed Forward Network
# ---------------------------------------------------------

class V3FeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.fc1(x)

        x = F.gelu(x)

        x = self.fc2(x)

        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# V3 Transformer Block
# ---------------------------------------------------------

class V3TransformerBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(
            n_embd
        )

        self.self_attention = V3MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = RMSNorm(
            n_embd
        )

        self.ffn = V3FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        attention_mask=None,
        past_key_value=None,
        use_cache=False
    ):

        # Pre-normalized self-attention.
        normalized_x = self.norm1(x)

        attention_output, present = (
            self.self_attention(
                normalized_x,
                attention_mask=attention_mask,
                past_key_value=past_key_value,
                use_cache=use_cache
            )
        )

        x = x + attention_output

        # Pre-normalized feed-forward network.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V3 Encoder-Only Prefix-LM
# ---------------------------------------------------------

class V3LegalLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layers,
        n_head,
        max_sequence_length,
        prefix_length,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.max_sequence_length = (
            max_sequence_length
        )

        self.prefix_length = (
            prefix_length
        )

        # Shared token embedding.
        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Learned positional embeddings.
        self.position_embedding = nn.Embedding(
            max_sequence_length,
            n_embd
        )

        # Single encoder-style Transformer stack.
        self.layers = nn.ModuleList([
            V3TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tied language-model head.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        labels=None,
        past_key_values=None,
        use_cache=False
    ):

        B, T = input_ids.shape

        # Determine the position offset during
        # autoregressive generation.
        if past_key_values is None:

            position_offset = 0

        else:

            position_offset = (
                past_key_values[0][0].size(-2)
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device
        )

        x = self.token_embedding(
            input_ids
        )

        x = x + self.position_embedding(
            positions
        ).unsqueeze(0)

        # During normal training we process the
        # complete prefix + target sequence.
        #
        # During generation, the prefix is first
        # processed with its bidirectional mask.
        # Afterwards the generated tokens use the
        # cached keys and values.

        attention_mask = None

        if past_key_values is None:

            attention_mask = create_prefix_lm_mask(
                sequence_length=T,
                prefix_length=min(
                    self.prefix_length,
                    T
                ),
                device=input_ids.device
            )

        presents = [] if use_cache else None

        for layer_index, layer in enumerate(
            self.layers
        ):

            past = None

            if past_key_values is not None:

                past = past_key_values[
                    layer_index
                ]

            x, present = layer(
                x,
                attention_mask=attention_mask,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:

                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1),
                ignore_index=-100
            )

        return (
            logits,
            loss,
            presents
        )


print("V3 Encoder-Only Prefix-LM model classes defined.") 

## 6. Load the trained V3 checkpoint and verify compatibility

In [ ]:
checkpoint_path=ASSETS["v3_model.pt"]
try:
    phase1_checkpoint=torch.load(checkpoint_path,map_location="cpu",weights_only=True)
except TypeError:
    phase1_checkpoint=torch.load(checkpoint_path,map_location="cpu")
if "model_state_dict" not in phase1_checkpoint or "config" not in phase1_checkpoint:
    raise ValueError("v3_model.pt does not have the expected model_state_dict/config structure")
V3_CONFIG=phase1_checkpoint["config"]
BASE_STATE=phase1_checkpoint["model_state_dict"]
assert V3_CONFIG.get("architecture")=="encoder_only_prefix_lm", V3_CONFIG.get("architecture")
assert int(V3_CONFIG["vocab_size"])==tokenizer.vocab_size
MAX_SEQUENCE_LENGTH=int(V3_CONFIG["source_length"])+int(V3_CONFIG["target_length"])

def create_backbone(layer_indices=None):
    n_layers=int(V3_CONFIG["n_layers"]) if layer_indices is None else len(layer_indices)
    backbone=V3LegalLM(vocab_size=int(V3_CONFIG["vocab_size"]),n_embd=int(V3_CONFIG["n_embd"]),n_layers=n_layers,n_head=int(V3_CONFIG["n_head"]),max_sequence_length=MAX_SEQUENCE_LENGTH,prefix_length=int(V3_CONFIG["source_length"]),ffn_multiplier=float(V3_CONFIG["ffn_multiplier"]),dropout=float(V3_CONFIG.get("dropout",0.0)))
    if layer_indices is None:
        backbone.load_state_dict(BASE_STATE,strict=True)
    else:
        mapped={}
        selected=list(layer_indices)
        for key,value in BASE_STATE.items():
            if key.startswith("layers."):
                parts=key.split("."); old_i=int(parts[1])
                if old_i in selected: mapped["layers."+str(selected.index(old_i))+"."+".".join(parts[2:])]=value
            else: mapped[key]=value
        backbone.load_state_dict(mapped,strict=True)
    return backbone

smoke_backbone=create_backbone()
base_parameter_count=sum(p.numel() for p in smoke_backbone.parameters())
print("V3 checkpoint loaded strictly; parameters:",f"{base_parameter_count:,}","(",f"{base_parameter_count/1e6:.2f}M", ")")
assert base_parameter_count>0
del smoke_backbone; gc.collect()


## 7. Encode each context as a bidirectional V3 prefix

In [ ]:
def v3_encode_prefix(backbone,input_ids):
    batch_size,seq_len=input_ids.shape
    if seq_len>int(V3_CONFIG["source_length"]):
        raise ValueError("Context window exceeds V3 bidirectional prefix length")
    positions=torch.arange(seq_len,device=input_ids.device)
    x=backbone.token_embedding(input_ids)+backbone.position_embedding(positions).unsqueeze(0)
    # Every non-padding token in this short input is in the bidirectional prefix.
    key_mask=input_ids.ne(PAD_TOKEN_ID)[:,None,None,:]
    for layer in backbone.layers:
        x,_=layer(x,attention_mask=key_mask,use_cache=False)
    return backbone.final_norm(x)


## 8. PEFT adapters and rhetorical-role classifier

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self,base,rank=ADAPTER_RANK,alpha=ADAPTER_ALPHA,dropout=ADAPTER_DROPOUT,rank_stabilized=False):
        super().__init__(); self.base=base; self.rank=rank
        self.scale=alpha/(math.sqrt(rank) if rank_stabilized else rank)
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Linear(base.in_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
        for p in self.base.parameters(): p.requires_grad=False
    def forward(self,x): return self.base(x)+self.B(self.A(self.dropout(x)))*self.scale

class LoRMAPlusLinear(nn.Module):
    # LoRMA+ applies (I + alpha/r * B A) to the original projection output.
    def __init__(self,base,rank=ADAPTER_RANK,alpha=ADAPTER_ALPHA,dropout=ADAPTER_DROPOUT):
        super().__init__(); self.base=base; self.rank=rank; self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Linear(base.out_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.kaiming_uniform_(self.A.weight,a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
        for p in self.base.parameters(): p.requires_grad=False
    def forward(self,x):
        y=self.base(x)
        return y+self.B(self.A(self.dropout(y)))*self.scale

def add_adapters(backbone,method):
    for layer in backbone.layers:
        attn=layer.self_attention
        for attr in ("q_proj","k_proj","v_proj","out_proj"):
            original=getattr(attn,attr)
            if method=="lora": wrapped=LoRALinear(original,rank_stabilized=False)
            elif method=="rslora": wrapped=LoRALinear(original,rank_stabilized=True)
            elif method=="lorma_plus": wrapped=LoRMAPlusLinear(original)
            else: raise ValueError(f"Unknown adapter method: {method}")
            setattr(attn,attr,wrapped)
    return backbone

class RRHead(nn.Module):
    def __init__(self,hidden_size,num_labels):
        super().__init__()
        self.domain_embedding=nn.Embedding(2,hidden_size)
        nn.init.zeros_(self.domain_embedding.weight)
        self.net=nn.Sequential(nn.LayerNorm(hidden_size),nn.Dropout(0.2),nn.Linear(hidden_size,384),nn.GELU(),nn.Dropout(0.2),nn.Linear(384,num_labels))
    def forward(self,x,domain_ids): return self.net(x+self.domain_embedding(domain_ids))

class V3RRClassifier(nn.Module):
    def __init__(self,backbone,num_labels):
        super().__init__(); self.backbone=backbone; self.head=RRHead(backbone.n_embd,num_labels)
    def forward(self,input_ids,focus_mask,domain_ids):
        hidden=v3_encode_prefix(self.backbone,input_ids)
        weights=focus_mask.unsqueeze(-1).to(hidden.dtype)
        pooled=(hidden*weights).sum(dim=1)/weights.sum(dim=1).clamp_min(1.0)
        return self.head(pooled,domain_ids)

# Check that both identity-initialized adapter families start as the original Linear layer.
probe=nn.Linear(32,32,bias=False); probe_input=torch.randn(2,5,32)
for wrapped in (LoRALinear(copy.deepcopy(probe)),LoRALinear(copy.deepcopy(probe),rank_stabilized=True),LoRMAPlusLinear(copy.deepcopy(probe))):
    assert torch.allclose(probe(probe_input),wrapped(probe_input),atol=1e-6)
print("Adapter initialization checks passed.")


## 9. Training and evaluation helpers

In [ ]:
# Mild square-root class weights are estimated only from the training labels.
counts=np.array([train_counts.get(i,0) for i in range(len(LABELS))],dtype=np.float64)
raw=np.sqrt(counts.sum()/np.maximum(counts,1.0))
raw=np.minimum(raw,4.0); raw=raw/raw.mean()
CLASS_WEIGHTS=torch.tensor(raw,dtype=torch.float32,device=DEVICE)
print("Class weights:",{LABELS[i]:round(float(CLASS_WEIGHTS[i]),3) for i in range(len(LABELS))})

def metrics_for(gold,pred):
    n=len(LABELS); cm=np.zeros((n,n),dtype=np.int64)
    for y,p in zip(gold,pred): cm[int(y),int(p)]+=1
    tp=np.diag(cm); support=cm.sum(axis=1); predicted=cm.sum(axis=0)
    precision=tp/np.maximum(predicted,1); recall=tp/np.maximum(support,1)
    f1=2*precision*recall/np.maximum(precision+recall,1e-12)
    supported=support>0
    return {"macro_precision_all":float(precision.mean()),"macro_recall_all":float(recall.mean()),"macro_f1_all":float(f1.mean()),"macro_f1_gold_supported":float(f1[supported].mean()) if supported.any() else 0.0,"gold_supported_label_count":int(supported.sum()),"support_by_label":{LABELS[i]:int(support[i]) for i in range(n)},"confusion_matrix":cm.tolist()}

def predict_rows(model,rows):
    loader=make_loader(rows,False); model.eval(); pred=[]
    with torch.no_grad():
        for batch in loader:
            ids=batch["input_ids"].to(DEVICE,non_blocking=True); focus=batch["focus_mask"].to(DEVICE,non_blocking=True); domains=batch["domain_ids"].to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP): logits=model(ids,focus,domains)
            pred.extend(logits.argmax(dim=-1).cpu().tolist())
    return pred

def evaluate_domains(model,rows):
    preds=predict_rows(model,rows); out={}; gold_by=defaultdict(list); pred_by=defaultdict(list)
    for row,p in zip(rows,preds): gold_by[row["domain"]].append(row["label"]); pred_by[row["domain"]].append(p)
    for domain in ("CL","IT"): out[domain]=metrics_for(gold_by[domain],pred_by[domain])
    out["mean_domain_macro_f1_all"]=float(np.mean([out[d]["macro_f1_all"] for d in ("CL","IT")]))
    return out,preds

def build_rr_model(adapter_method,student_layers=None):
    backbone=create_backbone(student_layers)
    for p in backbone.parameters(): p.requires_grad=False
    if adapter_method is not None: backbone=add_adapters(backbone,adapter_method)
    return V3RRClassifier(backbone,len(LABELS))

def trainable_state(model):
    return {k:v.detach().cpu().clone() for k,v in model.named_parameters() if v.requires_grad}

def load_trainable_state(model,state):
    parameters=dict(model.named_parameters())
    missing=[]
    with torch.no_grad():
        for name,value in state.items():
            if name not in parameters: missing.append(name); continue
            parameters[name].copy_(value.to(parameters[name].device,dtype=parameters[name].dtype))
    if missing: raise KeyError(f"Trainable checkpoint keys not found: {missing[:5]}")

def run_experiment(experiment_name,adapter_method,student_layers=None,teacher_logits=None):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if DEVICE.type=="cuda": torch.cuda.manual_seed_all(SEED)
    model=build_rr_model(adapter_method,student_layers).to(DEVICE)
    train_params=[p for p in model.parameters() if p.requires_grad]
    trainable=sum(p.numel() for p in train_params)
    print(f"\n{experiment_name}: trainable parameters {trainable:,}; layers={len(model.backbone.layers)}; adapter={adapter_method}")
    optimizer=torch.optim.AdamW(train_params,lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_score=-1.0; best_state=None; best_metrics=None; best_epoch=0; wait=0; history=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); start=time.time(); total_loss=0.0; batches=0
        for batch in TRAIN_LOADER:
            ids=batch["input_ids"].to(DEVICE,non_blocking=True); focus=batch["focus_mask"].to(DEVICE,non_blocking=True); domains=batch["domain_ids"].to(DEVICE,non_blocking=True); labels=batch["labels"].to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP):
                logits=model(ids,focus,domains)
                hard_loss=F.cross_entropy(logits.float(),labels,weight=CLASS_WEIGHTS,label_smoothing=LABEL_SMOOTHING)
                if teacher_logits is None: loss=hard_loss
                else:
                    soft=teacher_logits[batch["indices"]].to(DEVICE,non_blocking=True)
                    soft_loss=F.kl_div(F.log_softmax(logits.float()/KD_TEMPERATURE,dim=-1),F.softmax(soft/KD_TEMPERATURE,dim=-1),reduction="batchmean")*(KD_TEMPERATURE**2)
                    loss=(1.0-KD_SOFT_WEIGHT)*hard_loss+KD_SOFT_WEIGHT*soft_loss
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(train_params,GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            total_loss+=float(loss.detach()); batches+=1
        val_metrics,_=evaluate_domains(model,DEV_EXAMPLES)
        score=val_metrics["mean_domain_macro_f1_all"]
        row={"epoch":epoch,"train_loss":total_loss/max(batches,1),"mean_dev_macro_f1_all":score,"CL_macro_f1_all":val_metrics["CL"]["macro_f1_all"],"IT_macro_f1_all":val_metrics["IT"]["macro_f1_all"],"elapsed_seconds":time.time()-start}
        history.append(row); print(f"{experiment_name} epoch {epoch}/{EPOCHS} loss={row['train_loss']:.4f} CL={row['CL_macro_f1_all']:.4f} IT={row['IT_macro_f1_all']:.4f} mean={score:.4f}")
        if score>best_score:
            best_score=score; best_state=trainable_state(model); best_metrics=val_metrics; best_epoch=epoch; wait=0
        else:
            wait+=1
            if wait>=PATIENCE: print("Early stopping on development Macro-F1."); break
    path=OUTPUT_DIR/f"{experiment_name}_trainable.pt"
    torch.save({"experiment":experiment_name,"adapter_method":adapter_method,"student_layers":student_layers,"trainable_state":best_state,"best_epoch":best_epoch,"best_dev_metrics":best_metrics,"trainable_parameters":trainable},path)
    with open(OUTPUT_DIR/f"{experiment_name}_history.json","w",encoding="utf-8") as f: json.dump(history,f,indent=2)
    del model,optimizer,scaler; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return {"adapter_method":adapter_method,"student_layers":student_layers,"state":best_state,"metrics":best_metrics,"best_epoch":best_epoch,"trainable_parameters":trainable,"history":history,"checkpoint_path":str(path)}


## 10. Run the three matched PEFT comparisons

In [ ]:
PEFT_RESULTS={}
for method in ("lora","rslora","lorma_plus"):
    PEFT_RESULTS[method]=run_experiment(method,method)


## 11. Distill the strongest PEFT teacher into a smaller V3 student

In [ ]:
teacher_method=max(PEFT_RESULTS,key=lambda m:PEFT_RESULTS[m]["metrics"]["mean_domain_macro_f1_all"])
print("Teacher selected using development Macro-F1:",teacher_method)
TEACHER_LAYERS=list(range(int(V3_CONFIG["n_layers"])))
STUDENT_LAYER_INDICES=TEACHER_LAYERS[::2]
if STUDENT_LAYER_INDICES[-1] != TEACHER_LAYERS[-1]: STUDENT_LAYER_INDICES.append(TEACHER_LAYERS[-1])
print("Teacher layers:",len(TEACHER_LAYERS),"student layers:",STUDENT_LAYER_INDICES)

teacher=build_rr_model(teacher_method).to(DEVICE)
load_trainable_state(teacher,PEFT_RESULTS[teacher_method]["state"])
teacher.eval()
train_ordered_loader=make_loader(TRAIN_EXAMPLES,False)
teacher_logits=torch.empty((len(TRAIN_EXAMPLES),len(LABELS)),dtype=torch.float32)
with torch.no_grad():
    for batch in train_ordered_loader:
        ids=batch["input_ids"].to(DEVICE,non_blocking=True); focus=batch["focus_mask"].to(DEVICE,non_blocking=True); domains=batch["domain_ids"].to(DEVICE,non_blocking=True)
        with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP): logits=teacher(ids,focus,domains)
        teacher_logits[batch["indices"]]=logits.float().cpu()
print("Cached teacher logits:",tuple(teacher_logits.shape))
del teacher; gc.collect()
if DEVICE.type=="cuda": torch.cuda.empty_cache()

KD_RESULT=run_experiment("kd_student_6layer",teacher_method,student_layers=STUDENT_LAYER_INDICES,teacher_logits=teacher_logits)


## 12. Compare development results and save the run record

In [ ]:
ALL_RESULTS={**PEFT_RESULTS,"kd_student_6layer":KD_RESULT}
comparison=[]
for name,result in ALL_RESULTS.items():
    m=result["metrics"]
    comparison.append({"experiment":name,"adapter":result["adapter_method"],"layers":len(result["student_layers"]) if result["student_layers"] is not None else int(V3_CONFIG["n_layers"]),"best_epoch":result["best_epoch"],"trainable_parameters":result["trainable_parameters"],"CL_macro_f1_all":m["CL"]["macro_f1_all"],"IT_macro_f1_all":m["IT"]["macro_f1_all"],"mean_domain_macro_f1_all":m["mean_domain_macro_f1_all"],"CL_gold_supported_f1":m["CL"]["macro_f1_gold_supported"],"IT_gold_supported_f1":m["IT"]["macro_f1_gold_supported"]})
comparison=sorted(comparison,key=lambda x:x["mean_domain_macro_f1_all"],reverse=True)
for row in comparison: print(row)
with open(OUTPUT_DIR/"model_comparison.json","w",encoding="utf-8") as f: json.dump(comparison,f,indent=2)
with open(OUTPUT_DIR/"development_metrics_detailed.json","w",encoding="utf-8") as f: json.dump({name:r["metrics"] for name,r in ALL_RESULTS.items()},f,indent=2)


## 13. Generate test predictions and Codabench ZIPs

In [ ]:
def prediction_map(rows,preds):
    result={}
    for row,pred in zip(rows,preds):
        doc_id=row["doc_id"]
        if doc_id not in result: result[doc_id]=[None]*row["document_length"]
        result[doc_id][row["sentence_index"]]=ID_TO_LABEL[int(pred)]
    if any(any(x is None for x in labels) for labels in result.values()): raise ValueError("Some test sentences did not receive a prediction")
    return result

def validate_prediction_map(predictions):
    expected={str(d["id"]):len(d["text"]) for dom in ("CL","IT") for d in SPLITS[f"{dom}_test.json"]}
    if len(expected)!=sum(len(SPLITS[f"{dom}_test.json"]) for dom in ("CL","IT")): raise ValueError("Duplicate test IDs across CL and IT")
    if set(predictions)!=set(expected): raise ValueError(f"Test ID mismatch; missing={set(expected)-set(predictions)}, extra={set(predictions)-set(expected)}")
    for doc_id,labels in predictions.items():
        if len(labels)!=expected[doc_id]: raise ValueError(f"{doc_id}: output length {len(labels)} != sentence count {expected[doc_id]}")
        if not set(labels)<=set(LABELS): raise ValueError(f"{doc_id}: invalid label found")
    return True

def write_submission(predictions,name):
    validate_prediction_map(predictions)
    json_path=OUTPUT_DIR/f"{name}_prediction.json"
    zip_path=OUTPUT_DIR/f"{name}_submission.zip"
    with open(json_path,"w",encoding="utf-8") as f: json.dump(predictions,f,ensure_ascii=False,indent=2)
    with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf: zf.write(json_path,arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as zf: assert zf.namelist()==["prediction.json"]
    return zip_path

TEST_PREDICTIONS={}
for name,result in ALL_RESULTS.items():
    model=build_rr_model(result["adapter_method"],result["student_layers"]).to(DEVICE)
    load_trainable_state(model,result["state"]); model.eval()
    preds=predict_rows(model,TEST_EXAMPLES)
    TEST_PREDICTIONS[name]=prediction_map(TEST_EXAMPLES,preds)
    write_submission(TEST_PREDICTIONS[name],name)
    del model; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

best_name=comparison[0]["experiment"]
best_prediction=TEST_PREDICTIONS[best_name]
with open(OUTPUT_DIR/"prediction.json","w",encoding="utf-8") as f: json.dump(best_prediction,f,ensure_ascii=False,indent=2)
with zipfile.ZipFile(OUTPUT_DIR/"submission.zip","w",compression=zipfile.ZIP_DEFLATED) as zf: zf.write(OUTPUT_DIR/"prediction.json",arcname="prediction.json")
with zipfile.ZipFile(OUTPUT_DIR/"submission.zip") as zf: assert zf.namelist()==["prediction.json"]
with open(OUTPUT_DIR/"phase3_rr_predictions_by_domain.json","w",encoding="utf-8") as f:
    by_domain={dom:{str(d["id"]):TEST_PREDICTIONS[best_name][str(d["id"])] for d in SPLITS[f"{dom}_test.json"]} for dom in ("CL","IT")}
    json.dump(by_domain,f,ensure_ascii=False,indent=2)
print("Selected from development metrics:",best_name)
print("Final ZIP:",OUTPUT_DIR/"submission.zip")


## 14. Output checklist and methods cited

In [ ]:
print("Output directory:",OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file(): print(path.name, f"{path.stat().st_size/1024**2:.2f} MiB")
print("\nFinal submission structure:")
with zipfile.ZipFile(OUTPUT_DIR/"submission.zip") as zf: print(zf.namelist())
assert (OUTPUT_DIR/"prediction.json").exists()


### Methods and sources

- **LoRA**: additive low-rank updates to frozen weights (Hu et al., 2021), [paper](https://arxiv.org/abs/2106.09685).
- **rsLoRA**: uses rank-stabilized scaling, commonly expressed as alpha divided by the square root of rank (Kalajdzievski, 2023), [paper](https://arxiv.org/abs/2312.03732).
- **LoRMA+**: the multiplicative low-rank approach with identity-preserving additive rank inflation, as described in Bihany, Patel, and Modi (Findings of ACL 2025), [paper](https://aclanthology.org/2025.findings-acl.527/). This notebook compares LoRMA+ with the additive LoRA family using the same V3 projection layers.
- **Knowledge distillation**: the 11-layer selected PEFT teacher supplies soft class distributions; a 6-layer student receives both soft targets and gold labels (Hinton, Vinyals, and Dean, 2015), [paper](https://arxiv.org/abs/1503.02531).

Development-set comparisons are for model selection only. Codabench scores are not known until the ZIP is submitted, and a higher score is not guaranteed.
